Imports

In [ ]:
import json
import random
import time
import requests
import pandas as pd

Here are request params constants to use later. I also have separate bands of different trading volumes to try to get an even spread of markets with different volumes to show differences between high, medium and low markets.

In [ ]:
BASE = "https://gamma-api.polymarket.com"
PAGE_SIZE = 100
PER_BAND = 600
SLICE_DAYS = 7
POOL_PER_SLICE = 100
random.seed(42)
END_DATE_MIN = "2025-01-01"
END_DATE_MAX = "2026-10-31"

VOLUME_BANDS = [
    (100, 1_000),
    (1_000, 10_000),
    (10_000, 100_000),
    (100_000, 1_000_000),
    (1_000_000, None),
]

Just testing to see if the request works correctly.

In [ ]:
params = {"closed": "true", "order": "volumeNum", "ascending": "false", "limit": 10, "include_tag": "true"}
res = requests.get(f"{BASE}/markets", params=params, timeout=30)
sample = res.json()
print(f"Status: {res.status_code} | markets returned: {len(sample)}\n")

Should list all the fields that the response returns as well as if they are missing in any to look out for later, what the type is and an example value.

In [ ]:
all_keys = sorted({k for m in sample for k in m})
print(f"{len(all_keys)} fields across {len(sample)} markets\n")

rows = []
for k in all_keys:
    vals = [m[k] for m in sample if k in m]
    example = next((v for v in vals if v not in (None, "", 0, [], {})), vals[0] if vals else None)
    rows.append({
        "field": k,
        "present_in": f"{len(vals)}/{len(sample)}",
        "type": type(example).__name__,
        "example": str(example)[:80],
    })

fields = pd.DataFrame(rows)
pd.set_option("display.max_rows", None)
fields

Actually calls the API value and has some helper function. parse_json_list helps because for example outcomes comes in as '["yes", "no"]' which python treats as a string but I need to turn it into a list '["yes", "no"]' -> ["yes", 'no"].

The API used to have a broad field for "Categories" but got removed for a new Tag system. Broad category of "Sports" became "NFL, "NBA", "Basketball", etc. We can either keep it like it is or try to summarize it into a broader category later.

I created a main list of rows for the markets and saved a secondary list of rows for the tags that reference the market ID. Maybe for showing up much volume for each Tag or something else later.

In [ ]:
def parse_json_list(value):
    if isinstance(value, list):
        return value
    try:
        return json.loads(value) if value else []
    except (TypeError, ValueError):
        return []


def fetch_band(vmin, vmax):
    out, offset = [], 0
    while len(out) < PER_BAND:
        params = {
            "closed": "true",
            "volume_num_min": vmin,
            "end_date_min": END_DATE_MIN,
            "end_date_max": END_DATE_MAX,
            "order": "endDate",
            "ascending": "false",
            "limit": PAGE_SIZE,
            "offset": offset,
            "include_tag": "true",
        }
        if vmax is not None:
            params["volume_num_max"] = vmax
        res = requests.get(f"{BASE}/markets", params=params, timeout=30)
        res.raise_for_status()
        page = res.json()
        if not page:
            break
        out.extend(page)
        if len(page) < PAGE_SIZE:
            break
        offset += PAGE_SIZE          # move to the next page (prevents re-fetching the same page)
        time.sleep(0.3)
    return out[:PER_BAND]


rows, tag_rows = [], []
for vmin, vmax in VOLUME_BANDS:
    band_label = f"${vmin:,}+" if vmax is None else f"${vmin:,}–${vmax:,}"
    for m in fetch_band(vmin, vmax):
        outcomes = parse_json_list(m.get("outcomes"))
        prices = [float(p) for p in parse_json_list(m.get("outcomePrices"))]
        winner = outcomes[prices.index(max(prices))] if outcomes and len(prices) == len(outcomes) else None
        tags = [t.get("label") for t in (m.get("tags") or []) if t.get("label")]
        events = m.get("events") or [{}]
        question = m.get("question") or ""

        rows.append({
            "market_id": m.get("id"),
            "question": question,
            "event_title": events[0].get("title"),
            "start_date": m.get("startDate"),
            "end_date": m.get("endDate"),
            "volume_usd": m.get("volumeNum"),
            "volume_band": band_label,
            "tags": ", ".join(tags),          # topic labels exactly as the API provides them
            "tag_count": len(tags),
            "question_length": len(question),
            "num_outcomes": len(outcomes),
            "resolved_outcome": winner,
            "neg_risk": bool(m.get("negRisk")),   # True = part of a multi-option event (e.g. "Who will win...?")
            "slug": m.get("slug"),
        })
        for label in tags:
            tag_rows.append({"market_id": m.get("id"), "tag": label})

markets = pd.DataFrame(rows).drop_duplicates("market_id")

Here is a sample of the market

In [ ]:
#TODO worth to investigate more the quality of the markets / a wider date range because I think this mainly just gets values for recent days maybe a bunch of separate calls for random start/end dates

In [ ]:
print("Shape: ", markets.shape)
markets

Cleaned dates from string values to datetimes

In [ ]:
markets["start_date"] = pd.to_datetime(markets["start_date"], errors="coerce", utc=True).dt.tz_localize(None)
markets["end_date"] = pd.to_datetime(markets["end_date"], errors="coerce", utc=True).dt.tz_localize(None)

No rows are missing a volume field but might be depending on when the call is made so leaving here for now

In [ ]:
print("Rows missing volume_usd:", markets["volume_usd"].isna().sum())
markets = markets.dropna(subset=["start_date", "end_date", "volume_usd"])

When dates get figured out more might be interesting to show how long the market was open for

In [ ]:
markets["duration_days"] = (markets["end_date"] - markets["start_date"]).dt.days
markets = markets[markets["duration_days"] >= 0]
markets

Again when the dates get fixed this code would be more useful showing a trend of what months and days have higher volume

In [ ]:
markets["start_month"] = markets["start_date"].dt.to_period("M").dt.to_timestamp()
markets["start_weekday"] = markets["start_date"].dt.day_name()

markets

Here is a sample of the tags as well for later

In [ ]:
market_tags = pd.DataFrame(tag_rows, columns=["market_id", "tag"]).drop_duplicates()
market_tags = market_tags[market_tags["market_id"].isin(markets["market_id"])]

market_tags